# Importations

In [ ]:
!pip install langgraph langchain-openai langchain-community tavily-python requests langchain-google-genai

In [23]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [24]:
from google.colab import userdata
from langchain_google_genai import ChatGoogleGenerativeAI
import os
# 1. API Keys

gemini_api_key = userdata.get('GEMINI_API_KEY')
tavily_api_key = userdata.get('TAVILY_API_KEY')
# Use it in your environment or code


# Make sure to get a free API key from tavily.com
os.environ["GOOGLE_API_KEY"] = gemini_api_key
os.environ["TAVILY_API_KEY"] = tavily_api_key

llm = ChatGoogleGenerativeAI(
    model="gemini-3-flash-preview",
    temperature=0 # Keep it strict and factual for agricultural advice
)

# Agent 1

In [25]:
import torch
import torch.nn as nn
from torchvision import models, transforms
from PIL import Image
# If you are in Colab and haven't mounted drive yet, run this:
# from google.colab import drive
# drive.mount('/content/drive')

# 1. Setup Device and Class Names
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# These must match the exact alphabetical order from your training phase
class_names = [
    'Aphid', 'Black Rust', 'Blast', 'Brown Rust', 'Fusarium Head Blight',
    'Healthy Wheat', 'Leaf Blight', 'Mildew', 'Mite', 'Septoria',
    'Smut', 'Stem fly', 'Tan spot', 'Yellow Rust'
]
num_classes = len(class_names)

# 2. Recreate the Model Architecture
print("Loading EfficientNet-B3 architecture...")
model = models.efficientnet_b3(weights=None) # Set to None because we are loading our own weights
num_ftrs = model.classifier[1].in_features
model.classifier[1] = nn.Linear(num_ftrs, num_classes)

# 3. Load Your Saved Weights
model_path = '/content/drive/MyDrive/freelance_projects/wheat-AI-project/best_wheat_efficientnet.pth'
# map_location ensures it loads correctly whether you are currently using a GPU or CPU
model.load_state_dict(torch.load(model_path, map_location=device))
model = model.to(device)
model.eval() # CRITICAL: Set to evaluation mode
print("Model weights loaded successfully!")

# 4. Define the Exact Preprocessing (Must match training!)
preprocess = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.4616, 0.4831, 0.3297],
                         std=[0.2015, 0.1954, 0.1890])
])

# 5. Define the Prediction Function
def predict_wheat_disease(image_path):
    try:
        # Load and preprocess the image
        img = Image.open(image_path).convert('RGB')
        input_tensor = preprocess(img)

        # Add a batch dimension (models expect [batch_size, channels, height, width])
        input_batch = input_tensor.unsqueeze(0).to(device)

        # Run inference
        with torch.no_grad():
            output = model(input_batch)

        # Apply softmax to convert raw logits to probabilities (0.0 to 1.0)
        probabilities = torch.nn.functional.softmax(output[0], dim=0)

        # Get the highest probability and its corresponding class index
        confidence, predicted_idx = torch.max(probabilities, 0)

        predicted_class = class_names[predicted_idx.item()]
        confidence_pct = confidence.item() * 100

        print(f"\n--- Prediction Results ---")
        print(f"Image: {image_path}")
        print(f"Predicted Class: {predicted_class}")
        print(f"Confidence: {confidence_pct:.2f}%")

        return predicted_class, confidence_pct

    except Exception as e:
        print(f"Error processing image: {e}")

Using device: cpu
Loading EfficientNet-B3 architecture...
Model weights loaded successfully!


In [26]:
from langchain_core.tools import tool

@tool
def analyze_wheat_image(image_path: str) -> str:
    """
    Analyzes an image of a wheat crop to detect diseases or verify its health.
    Always use this tool when a user asks about the health, disease, or status of a wheat leaf image.

    Args:
        image_path (str): The absolute file path to the wheat image.

    Returns:
        str: A report of the predicted disease and the confidence score.
    """
    try:
        # We reuse the function we wrote earlier
        predicted_class, confidence = predict_wheat_disease(image_path)

        # Format the output so the LLM can easily read and converse about it
        if confidence > 80:
            return f"Analysis complete. I am highly confident ({confidence:.1f}%) that this plant shows signs of {predicted_class}."
        else:
            return f"Analysis complete. I predict this is {predicted_class}, but my confidence is somewhat low ({confidence:.1f}%). Recommend consulting an agronomist."

    except Exception as e:
        return f"Error analyzing the image: {str(e)}"

In [27]:
import os
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.prebuilt import create_react_agent
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_community.tools.tavily_search import TavilySearchResults


# 2. Define Tools
# (Assuming @tool analyze_wheat_image from previous step is loaded in memory)
tavily_tool = TavilySearchResults(max_results=3)

# Combine your custom PyTorch tool with the Web Search tool
diagnostic_agent_tools = [analyze_wheat_image, tavily_tool]

# 3. The Updated System Prompt (The "Bilan" Engine)
diagnostic_agent_system_message = SystemMessage(
    content="""You are an expert agricultural AI Diagnostic Agent specializing in wheat health.
Your goal is to provide a comprehensive diagnostic report (a 'Bilan') to farmers based on an uploaded image.

Workflow:
1. ALWAYS use the `analyze_wheat_image` tool first to get the predicted disease and confidence score.
2. If the prediction is a disease (not healthy), use the `tavily_search_results_json` tool to search for factual agricultural context about this specific wheat disease.
   -> Search specifically for: secondary visual symptoms, environmental causes, and risk/spread velocity.

Formatting your response:
Present your final answer as a structured Diagnostic Report using the following headers:
- 🔬 Initial Diagnosis & Confidence: (State the model's prediction)
- 👁️ Visual Confirmation: (What other symptoms should the farmer look for to verify this?)
- 🌧️ Environmental Causes: (What weather/humidity/temperature triggered this?)
- ⚠️ Risk Assessment: (How fast does this spread and what is the threat to yield?)

CRITICAL RULE: DO NOT provide any treatment, chemical, or mitigation advice. You are strictly a diagnostic agent."""
)

# 4. Build the Agent
diagnostic_agent = create_react_agent(llm, diagnostic_agent_tools)



/tmp/ipykernel_26373/2797631844.py:36: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  diagnostic_agent = create_react_agent(llm, diagnostic_agent_tools)


In [28]:
# # 5. Run the Agent
# user_input = "Please analyze this image and give me a full diagnostic report: /content/wheat_prepared/validation/Mite/mite_137.jpg"

# inputs = {"messages": [diagnostic_agent_system_message, HumanMessage(content=user_input)]}

# print("Agent is analyzing and researching...\n" + "-"*40)
# for chunk in diagnostic_agent.stream(inputs, stream_mode="values"):
#     message = chunk["messages"][-1]
#     message.pretty_print()

#Agent 2

In [29]:
import os
import requests

# 1. Define the Weather Tool
@tool
def get_agronomic_weather(latitude: float, longitude: float) -> dict:
    """
    Fetches real-time and 24-hour forecast environmental data for agricultural analysis.
    Always use this tool first to check the weather before recommending crop treatments,
    as rain, wind, or extreme temperatures dictate which treatments are viable.
    """
    url = "https://api.open-meteo.com/v1/forecast"
    params = {
        "latitude": latitude,
        "longitude": longitude,
        "current": "temperature_2m,relative_humidity_2m,precipitation,wind_speed_10m",
        "hourly": "relative_humidity_2m,soil_moisture_0_to_7cm",
        "timezone": "auto"
    }

    try:
        response = requests.get(url, params=params)
        response.raise_for_status()
        data = response.json()

        current = data.get("current", {})
        hourly = data.get("hourly", {})

        hum_list = [h for h in hourly.get("relative_humidity_2m", [])[:24] if h is not None]
        avg_hum = round(sum(hum_list) / len(hum_list), 1) if hum_list else None

        soil_list = [s for s in hourly.get("soil_moisture_0_to_7cm", [])[:24] if s is not None]
        avg_soil = round(sum(soil_list) / len(soil_list), 3) if soil_list else "Data Unavailable"

        return {
            "status": "success",
            "temperature_c": current.get("temperature_2m"),
            "rain_mm": current.get("precipitation"),
            "wind_kmh": current.get("wind_speed_10m"),
            "humidity_24h_avg_percent": avg_hum,
            "soil_moisture_24h_avg": avg_soil
        }
    except Exception as e:
        return {"status": "error", "message": str(e)}

In [30]:
# 4. The Updated Treatment System Prompt
treatment_agent_system_message = SystemMessage(
    content="""You are an expert Agricultural Treatment Agent.
Your goal is to provide a safe, actionable treatment plan based on a provided Diagnostic Report and the current local weather.

Workflow:
1. Extract the disease name from the user's Diagnostic Report.
2. ALWAYS use the `get_agronomic_weather` tool first to check current conditions for the provided coordinates.
3. Use the `tavily_search_results_json` tool to search for specific, modern agricultural treatments for the identified disease.
4. Synthesize the treatment plan, adapting it to the weather (e.g., "Do not spray fungicide today because high rain is expected").

Format your final response strictly into TWO distinct sections:

🚨 QUICK ACTION SUMMARY
Provide a 2-3 sentence summary of the absolute most critical immediate action the farmer must take right now. You MUST state whether it is currently safe to apply treatments based on the weather.

📄 FULL TREATMENT REPORT
- 🌡️ Field Conditions & Treatment Viability: (Detailed breakdown of the weather and how it affects the treatment).
- 🛡️ Immediate Interventions: (Specific chemical or biological sprays, dosages if available, and application rules).
- 🌱 Cultural & Long-Term Management: (Pruning, watering adjustments, crop rotation, tool sanitization).

Keep your advice highly professional, focused on crop yield protection, and easy for a farmer to read."""
)


treatment_agent_tools = [get_agronomic_weather, tavily_tool]

# 5. Build the Agent
treatment_agent = create_react_agent(llm, treatment_agent_tools)


/tmp/ipykernel_26373/2352428005.py:29: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  treatment_agent = create_react_agent(llm, treatment_agent_tools)


In [31]:
# # 6. Run the Agent!
# # We simulate the output from Agent 1, plus the farm's location (e.g., Tunis)
# diagnostic_input = """
# Here is the Diagnostic Bilan and my farm coordinates (Latitude: 36.8065, Longitude: 10.1815):

# Initial Diagnosis & Confidence:
# The uploaded image is classified as Aphid with 93.7% confidence.

# Visual Confirmation:
# Compare key visual signs on the plant with known symptoms of Aphid.

# Environmental Causes:
# Environmental triggers depend on the disease profile and local field conditions.

# Risk Assessment:
# Estimated risk level from model confidence: High.
# """

# inputs = {"messages": [system_message, HumanMessage(content=diagnostic_input)]}

# print("Treatment Agent is checking weather and researching protocols...\n" + "-"*50)
# for chunk in treatment_agent.stream(inputs, stream_mode="values"):
#     message = chunk["messages"][-1]
#     message.pretty_print()

# MasterAgent

Name of the Architecture: Supervisor Pattern

In [32]:
# ============================================================
# MASTER AGENT — Supervisor Pattern + Memory
# ============================================================
from typing import Literal, Annotated
from datetime import datetime
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from pydantic import BaseModel

# ── 1. State (TypedDict — required by LangGraph) ────────────
class AgentState(TypedDict):
    messages:          Annotated[list, add_messages]
    diagnostic_report: str
    treatment_report:  str
    next_step:         str
    detected_disease:  str   # ← ADD THIS


In [33]:
# ── 2. Helper: safely extract string content from a message ─
def extract_content(content) -> str:
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        return " ".join(
            block.get("text", "") if isinstance(block, dict) else str(block)
            for block in content
        )
    return str(content)

In [34]:
# ── 3. Router Schema ─────────────────────────────────────────
class RouterDecision(BaseModel):
    next: Literal["diagnostic", "treatment", "both", "direct"]
    reasoning: str

In [35]:
# ── 4. Supervisor Node ───────────────────────────────────────
SUPERVISOR_PROMPT = """You are a supervisor orchestrating a wheat crop AI assistant.
Based on the user's message, decide how to route the request:

- "diagnostic" → user provides an actual image path and wants it analyzed.
- "treatment"  → user already has a diagnosis and asks for treatment or cure.
- "both"       → user provides an image AND asks for treatment too.
- "direct"     → EVERYTHING ELSE: greetings, follow-up questions about a previous report, general questions, asking about capabilities, or any message without an actual image file path.

CRITICAL: Only route to "diagnostic" or "both" if the message contains a real file path (e.g. /content/image.jpg). Questions ABOUT a previous report or diagnosis are always "direct" — the agent has conversation memory.

Reply ONLY with valid JSON: {{"next": "...", "reasoning": "..."}}"""

def supervisor_node(state: AgentState) -> AgentState:
    last_user_msg = state["messages"][-1].content
    router_llm = llm.with_structured_output(RouterDecision)
    decision: RouterDecision = router_llm.invoke([
        SystemMessage(content=SUPERVISOR_PROMPT),
        HumanMessage(content=extract_content(last_user_msg))
    ])
    print(f"\n🧭 Supervisor → [{decision.next}] | {decision.reasoning}\n")
    return {**state, "next_step": decision.next}

In [36]:
# ── 5. Diagnostic Agent Node ─────────────────────────────────
def run_diagnostic_node(state: AgentState) -> AgentState:
    print("🔬 Running Diagnostic Agent...")

    # Check if there's actually an image path in the messages
    import re
    all_text = " ".join(
        extract_content(m.content) for m in state["messages"] if hasattr(m, "content")
    )
    has_image = bool(re.search(r"/[\w./]+\.(?:jpg|jpeg|png)", all_text, re.IGNORECASE))

    if not has_image:
        print("⚠️ No image found — returning guidance message.")
        msg = "Please provide an image path so I can analyze it. Example: /content/your_wheat_photo.jpg"
        return {
            **state,
            "detected_disease": "",
            "messages": state["messages"] + [AIMessage(content=msg)]
        }

    inputs = {"messages": [diagnostic_agent_system_message] + state["messages"]}
    result = diagnostic_agent.invoke(inputs)
    report = extract_content(result["messages"][-1].content)

    image_match = re.search(r"(/[\w./]+\.(?:jpg|jpeg|png))", all_text, re.IGNORECASE)
    disease = "Unknown"
    if image_match:
        try:
            predicted_class, _ = predict_wheat_disease(image_match.group(1))
            disease = predicted_class
        except Exception:
            pass

    return {
        **state,
        "diagnostic_report": report,
        "detected_disease":  disease,
        "messages": state["messages"] + [AIMessage(content=f"[Diagnostic Report]\n{report}")]
    }

In [37]:
# ── 6. Treatment Agent Node ──────────────────────────────────
DEFAULT_COORDS = "Latitude 36.8065, Longitude 10.1815 (Tunis, Tunisia)"

def run_treatment_node(state: AgentState) -> AgentState:
    print("💊 Running Treatment Agent...")

    context = ""
    if state.get("diagnostic_report"):
        context += f"Previous Diagnostic Report:\n{state['diagnostic_report']}\n\n"

    last_human = next(
        (extract_content(m.content) for m in reversed(state["messages"])
         if isinstance(m, HumanMessage)),
        ""
    )
    user_query = context + last_human

    enhanced_prompt = SystemMessage(content=(
        treatment_agent_system_message.content +
        f"\n\nIMPORTANT: If the user did not provide farm coordinates, "
        f"use these default coordinates: {DEFAULT_COORDS}. "
        "You MUST always call the `get_agronomic_weather` tool before recommending any treatment."
    ))

    inputs = {"messages": [enhanced_prompt, HumanMessage(content=user_query)]}
    result = treatment_agent.invoke(inputs)
    report = extract_content(result["messages"][-1].content)

    return {
        **state,
        "treatment_report": report,
        "messages": state["messages"] + [AIMessage(content=f"[Treatment Report]\n{report}")]
    }

In [38]:
# ── 7. Direct Answer Node ────────────────────────────────────
def direct_answer_node(state: AgentState) -> AgentState:
    print("💬 Answering directly...")

    # Count how many turns have happened to control intro behavior
    human_count = sum(1 for m in state["messages"] if isinstance(m, HumanMessage))

    system_content = """You are a specialized AI assistant for wheat crop health. You can:
        - Diagnose 14 wheat conditions from leaf images (diseases, pests, or healthy)
        - Provide weather-aware treatment plans based on the farmer's location
        - Generate full diagnostic + treatment reports

        Behavior rules:
        1. Be CONCISE. On greetings, give a short warm welcome (1-2 sentences) and briefly mention you help with wheat diagnosis. Do NOT list all 14 diseases unless explicitly asked.
        2. When the user says something vague like "I have a question", just ask "Sure, what would you like to know?" — do NOT pre-list topics or guess.
        3. Mention your wheat-only scope ONCE per session at most, not on every reply.
        4. Match the user's energy — short questions get short answers. Only go detailed when the user asks for detail.
        5. If the user asks about a previous report, answer based on conversation context — be specific, not generic."""

    response = llm.invoke([
        SystemMessage(content=system_content),
        *state["messages"]
    ])
    return {**state, "messages": state["messages"] + [response]}

In [39]:
# ── 8. Aggregator Node ───────────────────────────────────────
def aggregator_node(state: AgentState) -> AgentState:
    if state.get("next_step") == "direct":
        return state

    diag  = state.get("diagnostic_report", "")
    treat = state.get("treatment_report", "")
    if diag and treat:
        today = datetime.now().strftime("%B %d, %Y")
        summary = llm.invoke([
            SystemMessage(content=(
                f"You are an agricultural AI report generator. Today's date is {today}. "
                "Combine the diagnostic and treatment reports into one clean, structured final report. "
                "Use 'AI Agronomic Report' as the header — never imply a human consultant wrote this. "
                "Do NOT add a 'wheat-only' disclaimer. Keep the report professional but not bloated."
            )),
            HumanMessage(content=f"Diagnostic Report:\n{diag}\n\nTreatment Report:\n{treat}")
        ])
        final_text = f"📋 AI AGRONOMIC REPORT\n\n{extract_content(summary.content)}"
        return {**state, "messages": state["messages"] + [AIMessage(content=final_text)]}
    return state

In [40]:
# ── 9. Routing Functions ─────────────────────────────────────
def route_from_supervisor(state: AgentState) -> str:
    return state["next_step"]

HEALTHY_CLASS = "Healthy Wheat"

def route_after_diagnostic(state: AgentState) -> str:
    disease   = state.get("detected_disease", "")
    next_step = state["next_step"]

    # Auto-chain to treatment if disease detected, regardless of routing decision
    if next_step == "both" or (disease and disease != HEALTHY_CLASS):
        print(f"⚡ Auto-chaining to Treatment Agent (detected: {disease})")
        return "treatment"

    return "aggregator"

In [41]:
# ── 10. Build the Graph ──────────────────────────────────────
builder = StateGraph(AgentState)

builder.add_node("supervisor",  supervisor_node)
builder.add_node("diagnostic",  run_diagnostic_node)
builder.add_node("treatment",   run_treatment_node)
builder.add_node("direct",      direct_answer_node)
builder.add_node("aggregator",  aggregator_node)

builder.set_entry_point("supervisor")

builder.add_conditional_edges("supervisor", route_from_supervisor, {
    "diagnostic": "diagnostic",
    "treatment":  "treatment",
    "both":       "diagnostic",
    "direct":     "direct",
})

builder.add_conditional_edges("diagnostic", route_after_diagnostic, {
    "treatment":  "treatment",
    "aggregator": "aggregator",
})

builder.add_edge("treatment", "aggregator")
builder.add_edge("direct",    "aggregator")
builder.add_edge("aggregator", END)


In [42]:
# ── 11. Compile WITH Memory ──────────────────────────────────
memory = MemorySaver()
master_agent = builder.compile(checkpointer=memory)
print("✅ Master Agent compiled with memory!")


✅ Master Agent compiled with memory!


In [43]:
from datetime import datetime

def chat(thread_id: str = "farmer-session-1"):
    config = {"configurable": {"thread_id": thread_id}}
    print(f"🌾 Wheat AI Assistant — Session: [{thread_id}]")
    print("Type 'exit' to quit.\n" + "=" * 50)

    while True:
        user_input = input("👤 You: ").strip()
        if not user_input:
            continue
        if user_input.lower() in ("exit", "quit", "bye"):
            print("👋 Goodbye! Session saved.")
            break

        # ✅ Only pass the new message — checkpointer restores everything else
        final_state = master_agent.invoke(
            {"messages": [HumanMessage(content=user_input)]},
            config=config
        )
        answer = extract_content(final_state["messages"][-1].content)
        print(f"\n🤖 Agent:\n{answer}\n" + "-" * 50 + "\n")

# Start chatting
chat(thread_id="farm-session-42")



🌾 Wheat AI Assistant — Session: [farm-session-42]
Type 'exit' to quit.
👤 You: hi

🧭 Supervisor → [direct] | The user provided a simple greeting without any image path or specific request for diagnosis or treatment.

💬 Answering directly...

🤖 Agent:
Hello! I'm here to help you diagnose wheat crop issues and provide treatment plans. How can I assist you today?
--------------------------------------------------

👤 You: analyze this image /content/dd.jpg

🧭 Supervisor → [diagnostic] | The user provided a specific image file path (/content/dd.jpg) and requested an analysis, which matches the criteria for the diagnostic route.

🔬 Running Diagnostic Agent...

--- Prediction Results ---
Image: /content/dd.jpg
Predicted Class: Aphid
Confidence: 93.66%

--- Prediction Results ---
Image: /content/dd.jpg
Predicted Class: Aphid
Confidence: 93.66%
⚡ Auto-chaining to Treatment Agent (detected: Aphid)
💊 Running Treatment Agent...

🤖 Agent:
📋 AI AGRONOMIC REPORT

# AI Agronomic Report
**Date:** March 

KeyboardInterrupt: Interrupted by user

In [ ]:
# Save to file
png_data = master_agent.get_graph().draw_mermaid_png()
with open("master_agent_graph.png", "wb") as f:
    f.write(png_data)
print("Graph saved to master_agent_graph.png")